In [1]:
import os
print(os.getcwd())


/Users/maceli/ifood_cs/code/notebook


In [2]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.append('..')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy.stats import mannwhitneyu, ttest_ind

import utilities.functions as functions

from utilities.graficos import plot_metricas



from utilities.functions import (
    gerar_stats,
    pedidos_group,
    criacao_ordens,
    conversao_imediata,
   
)

from utilities.testes_estatisticos import testes,teste_proporcao_por_janela

In [3]:
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 
df = pd.read_parquet(BASE_PATH / "gold" / "df_publico.parquet")

In [4]:
df.head()

,customer_id,is_target,active,created_at,delivery_address_state,merchant_id,order_created_at,order_id,order_total_amount,origin_platform,order_created_month,unique_order_hash,weekday,hour,day,total_amount_mes,ticket_medio,num_pedidos_mes,num_pedidos_hist,outlier_iqr,outlier_zscore,outlier_mad,lim_inf_iqr,lim_sup_iqr,zscore,mad_score,id_p99,id_p10
0,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,True,2018-04-06T02:48:42.887Z,SP,e80a8d7a13b52ca9ccf247ca3396f1018be41a5b014d79...,2019-01-11 14:07:48+00:00,c9a236fc0b0255433d0852dc90ef8b9949f3a3ab67e553...,10.00,ANDROID,1,3171095530380566075,Friday,14,11,206.50,12.15,17,19,False,False,False,0.01,104.10,-1.06,-1.36,0,1
1,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,True,2018-04-06T02:48:42.887Z,SP,05291eacc3cb88847b56af15838a0e653a1fcda54a1d88...,2019-01-05 13:38:28+00:00,db3b910b374ce240d551b829b297b9547f4e2feee9fd78...,12.00,ANDROID,1,14445504651562908569,Saturday,13,5,206.50,12.15,17,19,False,False,False,0.01,104.10,-1.01,-1.27,0,1
2,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,True,2018-04-06T02:48:42.887Z,SP,05291eacc3cb88847b56af15838a0e653a1fcda54a1d88...,2019-01-13 13:50:06+00:00,a20903719abdaae9f6639681845f95963de83e0047ff13...,13.00,ANDROID,1,16546632448525009961,Sunday,13,13,206.50,12.15,17,19,False,False,False,0.01,104.10,-0.98,-1.23,0,1
3,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,True,2018-04-06T02:48:42.887Z,SP,05291eacc3cb88847b56af15838a0e653a1fcda54a1d88...,2019-01-26 13:15:53+00:00,90652cc196fbfa352b1d4d61b18b5babf39fad949c7ccc...,13.00,ANDROID,1,16333679914791111686,Saturday,13,26,206.50,12.15,17,19,False,False,False,0.01,104.10,-0.98,-1.23,0,1
4,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,True,2018-04-06T02:48:42.887Z,SP,05291eacc3cb88847b56af15838a0e653a1fcda54a1d88...,2019-01-15 13:48:33+00:00,9091a0ead77e2ed0d5ed9ce6787631386bd120036ff5b7...,13.00,ANDROID,1,6333204433002785330,Tuesday,13,15,206.50,12.15,17,19,False,False,False,0.01,104.10,-0.98,-1.23,0,1


In [5]:
#df=df.head(1000)

In [6]:
df_stats_mes = (
        df.groupby(['order_created_month','is_target'])
          .agg(
              total_clientes=('customer_id', 'nunique')))

df_stats_mes  

total_clientes
order_created_month is_target                
1                   control            241457
                    target             312919
12                  control            241457
                    target             312919

In [7]:


# Monkey patch completo
plt.ioff()  # Desativa modo interativo

# Salva a referência original
_original_show = plt.show
_original_ion = plt.ion
_original_draw = plt.draw
_original_pause = plt.pause

# Substitui todas as funções que podem mostrar gráficos
plt.show = lambda *args, **kwargs: None
plt.ion = lambda *args, **kwargs: None
plt.draw = lambda *args, **kwargs: None
plt.pause = lambda *args, **kwargs: None

# Executa seu código
vars = ['hour', 'weekday', 'day', 'delivery_address_state']
resultados = {}

for v in vars:
    print(v)
    
    resultados[v] = gerar_stats(df, ['is_target', v])
    
    plot_metricas(
        df=resultados[v],
        eixo_x=v,
        metrics={
            'perc_amount': '% Valor',
            'perc_pedidos': '% Pedidos',
            'perc_clientes': '% Clientes'
        },
        highlight_metrics=['perc_amount']
    )
    
    file_path = os.path.join('../../Resultados/graficos', f"grafico_{v}.png")
    plt.savefig(file_path, dpi=300, bbox_inches="tight")
    plt.close('all')  # Fecha tudo

# Restaura as funções originais se necessário depois
# plt.show = _original_show
# plt.ion = _original_ion
# plt.draw = _original_draw
# plt.pause = _original_pause

hour
weekday
day
delivery_address_state


In [8]:
df_analise_completa = (
        df.groupby(['order_created_month', 'customer_id'])
        .agg(
            total_pedidos=('unique_order_hash', 'count'),
            total_amount=('order_total_amount', 'sum')
        )
        .reset_index()
        .assign(
            categoria_pedidos=lambda x: x['total_pedidos']
                .apply(lambda y: '1_pedido' if y == 1 else '2+_pedidos')
        )
        .groupby(['order_created_month', 'total_pedidos'])
        .agg(
            total_clientes=('customer_id', 'nunique'),
            #total_pedidos=('total_pedidos', 'sum'),
            total_amount=('total_amount', 'sum'),
            #avg_amount_por_cliente=('total_amount', 'mean')
       )
        .reset_index()
    )
df_analise_completa.head()

,order_created_month,total_pedidos,total_clientes,total_amount
0,1,1,195166,"9,282,452.04"
1,1,2,106593,"10,342,216.05"
2,1,3,67767,"9,756,996.35"
3,1,4,46341,"8,934,690.64"
4,1,5,32575,"7,874,521.82"


Numero de pedidos por cliente exatamente um ou mais de um

In [9]:
pedidos_group(df,group_vars='is_target')

,order_created_month,categoria_pedidos,is_target,total_clientes,total_pedidos,total_amount,avg_amount_por_cliente,perc_clientes,perc_pedidos,perc_amount
0,1,1_pedido,control,97595,97595,"4,631,471.85",47.46,40.42,11.55,11.42
1,1,1_pedido,target,97571,97571,"4,650,980.19",47.67,31.18,8.03,7.99
2,1,2+_pedidos,control,143862,747295,"35,938,996.42",249.82,59.58,88.45,88.58
3,1,2+_pedidos,target,215348,1116783,"53,524,934.71",248.55,68.82,91.97,92.01
4,12,1_pedido,control,139600,139600,"6,711,046.32",48.07,57.82,27.12,27.25
5,12,1_pedido,target,160812,160812,"7,760,006.02",48.26,51.39,22.34,22.65
6,12,2+_pedidos,control,101857,375066,"17,920,032.44",175.93,42.18,72.88,72.75
7,12,2+_pedidos,target,152107,558982,"26,505,286.97",174.25,48.61,77.66,77.35


In [10]:
df, dias_0=criacao_ordens(df,group_vars=['is_target','order_created_month'])
print('Media da diferenca em dias entre a primeira e segunda ordem, nao levando em consideracao o periodo do mes',dias_0)

Media da diferenca em dias entre a primeira e segunda ordem, nao levando em consideracao o periodo do mes 5.802379096308715


Dias entre a primeira ordem e segunda

In [11]:
summary_mes = gerar_stats(
    df[(df['rank_month'] == 2)&(df['order_created_month']==12)],
    ['days_since_first_order_month','is_target']
)


In [12]:
summary_mes[summary_mes['is_target']=='target'].head(10)

,order_created_month,days_since_first_order_month,is_target,total_clientes,total_pedidos,total_ordem,perc_clientes,perc_pedidos,perc_amount
0,12,0,target,23415,23415,"1,049,863.45",9.22,9.22,8.58
1,12,1,target,19015,19015,"884,087.51",7.49,7.49,7.23
3,12,2,target,13759,13759,"651,399.58",5.42,5.42,5.32
5,12,3,target,11190,11190,"538,865.25",4.41,4.41,4.40
6,12,4,target,9731,9731,"474,980.99",3.83,3.83,3.88
7,12,6,target,9627,9627,"472,435.25",3.79,3.79,3.86
9,12,7,target,9058,9058,"448,091.84",3.57,3.57,3.66
10,12,5,target,8951,8951,"435,555.79",3.52,3.52,3.56
14,12,8,target,6296,6296,"305,304.39",2.48,2.48,2.50
17,12,9,target,4882,4882,"234,990.60",1.92,1.92,1.92


In [13]:
janelas = [(0, 1), (0, 3), (0, 4), (10, 16)]

df_base_final = None
todos_resumos = []

for window_start, window_end in janelas:

    resumo, df_p = conversao_imediata(
        df,
        mes_base=12,
        r_ordem=2,
        window_start=window_start,
        window_end=window_end,
        fill_value=-999
    )

    # guardar resumo de cada iteração
    todos_resumos.append(resumo)

    # nome da coluna que a função CRIA
    col_origem = f"converteu{window_start}-{window_end}d"
    # nome da coluna que você quer na base final
    col_destino = f"converteu_{window_start}_{window_end}"

    # cria a nova coluna usando a coluna dinâmica da função
    df_p[col_destino] = df_p[col_origem]

    # montar/atualizar base final mantendo target e valor
    if df_base_final is None:
        df_base_final = df_p[
            ['customer_id', 'is_target', 'order_total_amount', col_destino]
        ].copy()
    else:
        df_base_final = df_base_final.merge(
            df_p[['customer_id', col_destino]],
            on='customer_id',
            how='left'
        )

# preencher NaN das flags com 0
for window_start, window_end in janelas:
    col_destino = f"converteu_{window_start}_{window_end}"
    df_base_final[col_destino] = df_base_final[col_destino].fillna(0).astype(int)

# resumo com TODAS as iterações
df_resumo_final = pd.concat(todos_resumos, ignore_index=True)


In [14]:
df_resumo_final

,is_target,total_clientes,clientes_convertidos,total_amount,total_amount_convertido,taxa_conversao,window_range,mes_base
0,control,241457,28418,"4,986,605.95","1,448,563.12",11.77,0-1d,12
1,target,312919,42430,"7,249,066.19","1,933,950.96",13.56,0-1d,12
2,control,241457,45195,"4,986,605.95","2,250,937.13",18.72,0-3d,12
3,target,312919,67379,"7,249,066.19","3,124,215.79",21.53,0-3d,12
4,control,241457,51524,"4,986,605.95","2,556,962.49",21.34,0-4d,12
5,target,312919,77110,"7,249,066.19","3,599,196.78",24.64,0-4d,12
6,control,241457,16619,"4,986,605.95","799,715.72",6.88,10-16d,12
7,target,312919,24904,"7,249,066.19","1,209,393.06",7.96,10-16d,12


In [15]:
teste_proporcao_por_janela(df_base_final)

,janela,total_control,conv_control,total_target,conv_target,taxa_control,taxa_target,z_stat,p_value,significativo_5%
0,converteu_0_1,241457,28418,312919,42430,11.77,13.56,19.79,0.00,True
1,converteu_0_3,241457,45195,312919,67379,18.72,21.53,25.83,0.00,True
2,converteu_0_4,241457,51524,312919,77110,21.34,24.64,28.89,0.00,True
3,converteu_10_16,241457,16619,312919,24904,6.88,7.96,15.09,0.00,True


In [16]:
ranges_personalizados = [
    (0, 1),    
    (1,2),
    (0,5),
    (5, 7),   
    (15,16),
    (20, 30),  
    (5, 6)    
]

df_resultados = testes(
    df, 
    ranges_janelas=ranges_personalizados
)

In [17]:
df_resultados

,is_target,total_clientes,clientes_convertidos,total_amount,total_amount_convertido,taxa_conversao,window_range,mes_base,z_stat_proporcao,p_value_proporcao,significativo_proporcao,mannwhitney_pvalue,ttest_welch_pvalue,cohen_d,significativo_valores,janela_inicio,janela_fim,janela_range
0,control,241457,28418,"4,986,605.95","1,448,563.12",11.77,0-1d,12,NaN,NaN,False,NaN,NaN,NaN,False,0,1,0-1d
1,target,312919,42430,"7,249,066.19","1,933,950.96",13.56,0-1d,12,NaN,NaN,False,NaN,NaN,NaN,False,0,1,0-1d
2,control,241457,21979,"4,986,605.95","1,030,171.03",9.10,1-2d,12,NaN,NaN,False,NaN,NaN,NaN,False,1,2,1-2d
3,target,312919,32774,"7,249,066.19","1,535,487.09",10.47,1-2d,12,NaN,NaN,False,NaN,NaN,NaN,False,1,2,1-2d
4,control,241457,57601,"4,986,605.95","2,851,183.31",23.86,0-5d,12,NaN,NaN,False,NaN,NaN,NaN,False,0,5,0-5d
5,target,312919,86061,"7,249,066.19","4,034,752.57",27.50,0-5d,12,NaN,NaN,False,NaN,NaN,NaN,False,0,5,0-5d
6,control,241457,18609,"4,986,605.95","906,752.52",7.71,5-7d,12,NaN,NaN,False,NaN,NaN,NaN,False,5,7,5-7d
7,target,312919,27636,"7,249,066.19","1,356,082.88",8.83,5-7d,12,NaN,NaN,False,NaN,NaN,NaN,False,5,7,5-7d
8,control,241457,3159,"4,986,605.95","149,700.96",1.31,15-16d,12,NaN,NaN,False,NaN,NaN,NaN,False,15,16,15-16d
9,target,312919,4753,"7,249,066.19","228,638.63",1.52,15-16d,12,NaN,NaN,False,NaN,NaN,NaN,False,15,16,15-16d
